# STO-MILP v10 — 8-Case Experimental Matrix

Per STO_MILP_Engineering_Spec_Final_v10.

**Modules**: Inter-day SOC (Method 1), Green SOC, RE20 + T-REC, PWL degradation,
PV routing (no export), segmented billing (κ=1.0035), terminal SOC band.

**Cases**: M0_I0_R0, M1_I0_R0, M2_I0_R0, M2_I1_R0 (mainline),
M2_I1_R1_p3, M2_I1_R1_p5, M2_I1_R2_p3, M2_I1_R2_p5

In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import time
import json
from pathlib import Path

from milp_common import get_config, load_data, CASE_TABLE, format_results

CFG = get_config()
print(f"\nOutput dir: {CFG['output_dir']}")

CRF PV (20yr, r=0.05): 0.0802
CRF BESS (15yr, r=0.05): 0.0963

Output dir: ../milp_outputs


## Model Builder

Builds the two-stage stochastic MILP with all v10 modules.

In [2]:
def build_and_solve(CFG, repday_data, calendar_order, info, case_flags):
    """Build and solve the v10 MILP for one case.
    
    Returns: dict with capacities, objective, RE%, T-REC cost, solve time.
    """
    use_method1 = case_flags.get('method1', True) and calendar_order is not None
    n_repdays = info['n_repdays']
    n_scenarios = info['n_scenarios']
    n_hours = info['n_hours']
    N = n_repdays * n_scenarios * n_hours  # total dispatch points
    PV_RATING = CFG['pv_rating_kw']
    
    # ── Pre-compute flat arrays ──
    pv_coeff = np.empty(N)
    load_arr = np.empty(N)
    pw_arr   = np.empty(N)  # prob * weight
    tou_arr  = np.empty(N)
    month_arr = np.empty(N, dtype=int)
    
    # Index mapping
    def idx(d, s, t):
        return d * (n_scenarios * n_hours) + s * n_hours + t
    
    for d in range(n_repdays):
        dd = repday_data[d]
        for s in range(n_scenarios):
            sc = dd['scenarios'][s]
            start = idx(d, s, 0)
            sl = slice(start, start + n_hours)
            pv_coeff[sl] = sc['pv_kw'] / PV_RATING
            load_arr[sl] = sc['load_kw']
            pw_arr[sl] = sc['prob'] * dd['weight']
            tou_arr[sl] = dd['tou']
            month_arr[sl] = dd['month']
    
    # First-hour indices for SOC reset (per repday-scenario)
    is_first = np.zeros(N, dtype=bool)
    for d in range(n_repdays):
        for s in range(n_scenarios):
            is_first[idx(d, s, 0)] = True
    first_hours = np.where(is_first)[0]
    later_hours = np.where(~is_first)[0]
    prev_idx_arr = np.arange(N) - 1
    
    # ── Build Gurobi model ──
    m = gp.Model('milp_v10')
    m.Params.OutputFlag = 1
    m.Params.TimeLimit = CFG['time_limit']
    m.Params.Method = 2       # Barrier
    m.Params.Presolve = 2     # Aggressive
    m.Params.Cuts = 2         # Aggressive
    m.Params.MIPGap = 1e-4    # 0.01%
    m.Params.MIPFocus = 1     # Feasible solutions first
    
    # ── Stage 1: Capacity variables ──
    cap_pv = m.addVar(ub=CFG['pv_max_kw'], name='cap_pv')
    cap_bp = m.addVar(ub=CFG['bess_p_max_kw'], name='cap_bp')
    cap_be = m.addVar(ub=CFG['bess_e_max_kwh'], name='cap_be')
    cap_cd = m.addVar(name='cap_cd')  # contract demand
    
    # ── Stage 2: Dispatch variables (N = repdays × scenarios × hours) ──
    p_grid    = m.addMVar(N, name='grid')       # grid import
    p_pv_load = m.addMVar(N, name='pv_load')    # PV → load
    p_pv_ch   = m.addMVar(N, name='pv_ch')      # PV → BESS charge
    p_curt    = m.addMVar(N, name='curt')        # PV curtailed
    p_ch      = m.addMVar(N, name='ch')          # total BESS charge
    p_dis     = m.addMVar(N, name='dis')         # BESS discharge
    bin_ch    = m.addMVar(N, vtype=GRB.BINARY, name='bch')
    bin_dis   = m.addMVar(N, vtype=GRB.BINARY, name='bdis')
    
    # Green SOC tracking
    green_e   = m.addMVar(N, name='green_e')     # green energy in BESS
    green_dis = m.addMVar(N, name='green_dis')   # green portion of discharge
    
    # Intra-day SOC (delta from start-of-day)
    delta_e_cum = m.addMVar(N, lb=-GRB.INFINITY, name='dec')  # cumulative ΔE
    
    # T-REC
    trec_buy = m.addVar(name='trec')  # T-REC purchase (kWh)
    
    m.update()
    
    eff_ch = CFG['eff_charge']
    eff_dis_inv = 1.0 / CFG['eff_discharge']
    
    # ── C1: Power balance (no export) ──
    # PV_to_load + grid + discharge = load + charge
    m.addConstrs(
        (p_pv_load[i] + p_grid[i] + p_dis[i] == load_arr[i] + p_ch[i]
         for i in range(N)), name='bal')
    
    # ── C2: PV availability ──
    # PV_to_load + PV_to_charge + curtail = PV_available
    m.addConstrs(
        (p_pv_load[i] + p_pv_ch[i] + p_curt[i] == pv_coeff[i] * cap_pv
         for i in range(N)), name='pv_avail')
    
    # PV_to_charge <= total charge (BESS can also charge from grid)
    m.addConstrs((p_pv_ch[i] <= p_ch[i] for i in range(N)), name='pv_ch_lim')
    
    # ── C3-C4: Charge/discharge limits + mutual exclusivity ──
    m.addConstrs((p_ch[i]  <= cap_bp * bin_ch[i]  for i in range(N)), name='ch_lim')
    m.addConstrs((p_dis[i] <= cap_bp * bin_dis[i] for i in range(N)), name='dis_lim')
    m.addConstrs((bin_ch[i] + bin_dis[i] <= 1 for i in range(N)), name='excl')
    
    # ── C5-C9: SOC via intra-day cumulative ΔE ──
    # delta_e_cum[first] = eff_ch * p_ch - eff_dis_inv * p_dis
    m.addConstrs(
        (delta_e_cum[i] == eff_ch * p_ch[i] - eff_dis_inv * p_dis[i]
         for i in first_hours), name='dec_init')
    # delta_e_cum[later] = delta_e_cum[prev] + eff_ch * p_ch - eff_dis_inv * p_dis
    m.addConstrs(
        (delta_e_cum[i] == delta_e_cum[prev_idx_arr[i]] + eff_ch * p_ch[i] - eff_dis_inv * p_dis[i]
         for i in later_hours), name='dec_cont')
    
    # ── Method 1: Inter-day SOC ──
    if use_method1:
        n_cal = len(calendar_order)
        E_inter = m.addMVar(n_cal, lb=0, name='Eint')  # inter-day SOC level
        
        # SOC bounds: soc_min * E_B <= E_inter[i] + delta_e_cum[d,s,t] <= soc_max * E_B
        for cal_idx, cal_day in enumerate(calendar_order):
            d = cal_day['d_idx']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    flat = idx(d, s, t)
                    m.addConstr(E_inter[cal_idx] + delta_e_cum[flat] >= CFG['soc_min'] * cap_be,
                                name=f'soc_lo_{cal_idx}_{s}_{t}')
                    m.addConstr(E_inter[cal_idx] + delta_e_cum[flat] <= CFG['soc_max'] * cap_be,
                                name=f'soc_hi_{cal_idx}_{s}_{t}')
        
        # Inter-day transition: E_inter[i+1] = E_inter[i] + E[expected ΔE over the day]
        for cal_idx in range(n_cal - 1):
            d = calendar_order[cal_idx]['d_idx']
            # Expected end-of-day ΔE = sum_s pi_s * delta_e_cum[d,s,23]
            expected_de = gp.LinExpr()
            for s in range(n_scenarios):
                prob_s = repday_data[d]['scenarios'][s]['prob']
                last_hour_idx = idx(d, s, n_hours - 1)
                expected_de += prob_s * delta_e_cum[last_hour_idx]
            m.addConstr(E_inter[cal_idx + 1] == E_inter[cal_idx] + expected_de,
                        name=f'inter_{cal_idx}')
        
        # Terminal band: |E_inter[last] - E_inter[0]| <= epsilon * E_B
        eps = CFG['epsilon_term']
        # Expected ΔE for last day
        d_last = calendar_order[-1]['d_idx']
        expected_de_last = gp.LinExpr()
        for s in range(n_scenarios):
            prob_s = repday_data[d_last]['scenarios'][s]['prob']
            last_idx = idx(d_last, s, n_hours - 1)
            expected_de_last += prob_s * delta_e_cum[last_idx]
        E_end = E_inter[n_cal - 1] + expected_de_last
        m.addConstr(E_end - E_inter[0] <=  eps * cap_be, name='term_hi')
        m.addConstr(E_end - E_inter[0] >= -eps * cap_be, name='term_lo')
        
        # Initial SOC
        m.addConstr(E_inter[0] == CFG['soc_init'] * cap_be, name='soc_init_inter')
    else:
        # No Method 1: daily SOC reset (traditional)
        # SOC = soc_init * E_B + delta_e_cum
        for i in range(N):
            m.addConstr(CFG['soc_init'] * cap_be + delta_e_cum[i] >= CFG['soc_min'] * cap_be,
                        name=f'soc_lo_{i}')
            m.addConstr(CFG['soc_init'] * cap_be + delta_e_cum[i] <= CFG['soc_max'] * cap_be,
                        name=f'soc_hi_{i}')
    
    # ── Green SOC tracking (linearized) ──
    # green_e tracks green energy in BESS
    # green_e[first] = soc_init_green_frac * E_B + eff_ch * p_pv_ch - green_dis
    # For simplicity, assume initial green fraction = 0.5 (half of initial SOC is green)
    green_init_frac = 0.5 * CFG['soc_init']  # fraction of E_B that starts green
    
    m.addConstrs(
        (green_e[i] == green_init_frac * cap_be + eff_ch * p_pv_ch[i] - green_dis[i]
         for i in first_hours), name='ge_init')
    m.addConstrs(
        (green_e[i] == green_e[prev_idx_arr[i]] + eff_ch * p_pv_ch[i] - green_dis[i]
         for i in later_hours), name='ge_cont')
    
    # green_dis <= total discharge
    m.addConstrs((green_dis[i] <= p_dis[i] for i in range(N)), name='gd_lim')
    # green_e >= 0 (already via lb=0)
    # green_e <= total SOC (approximate: green_e <= soc_max * E_B)
    m.addConstrs((green_e[i] <= CFG['soc_max'] * cap_be for i in range(N)), name='ge_cap')
    
    # ── C10-C11: Monthly billing with kappa ──
    months_in_data = sorted(set(month_arr))
    n_months = len(months_in_data)
    D_max = {mo: m.addVar(name=f'Dmax_{mo}') for mo in months_in_data}
    oc_seg1 = {mo: m.addVar(name=f'oc1_{mo}') for mo in months_in_data}
    oc_seg2 = {mo: m.addVar(name=f'oc2_{mo}') for mo in months_in_data}
    
    kappa = CFG['kappa']
    for i in range(N):
        mo = int(month_arr[i])
        m.addConstr(D_max[mo] >= kappa * p_grid[i], name=f'dmax_{mo}_{i}')
    
    for mo in months_in_data:
        # oc_seg1 = min(D_max - cap_cd, 0.10 * cap_cd) if D_max > cap_cd
        # oc_seg2 = max(D_max - 1.10 * cap_cd, 0)
        m.addConstr(oc_seg1[mo] >= D_max[mo] - cap_cd, name=f'oc1a_{mo}')
        m.addConstr(oc_seg1[mo] <= 0.10 * cap_cd, name=f'oc1b_{mo}')
        m.addConstr(oc_seg2[mo] >= D_max[mo] - 1.10 * cap_cd, name=f'oc2_{mo}')
    
    # ── C12-C13: RE20 + T-REC + Green SOC ──
    # RE = PV self-use + green discharge
    total_load = float(np.sum(pw_arr * load_arr))
    re_expr = gp.quicksum(
        pw_arr[i] * (p_pv_load[i] + green_dis[i]) for i in range(N))
    
    # RE + T-REC >= target * load
    m.addConstr(re_expr + trec_buy >= CFG['re_target'] * total_load, name='RE20')
    
    # ── Objective ──
    # Investment annuity
    inv_cost = (
        cap_pv * CFG['capex_pv_per_kw'] * CFG['crf_pv']
        + cap_bp * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
        + cap_be * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                    + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate'])
        + cap_cd * CFG['contract_price_per_kw_month'] * 12
    )
    
    # Operating cost: grid energy purchase
    grid_cost = gp.quicksum(pw_arr[i] * p_grid[i] * float(tou_arr[i]) for i in range(N))
    
    # Billing penalties (monthly)
    bill_penalty = gp.quicksum(
        oc_seg1[mo] * CFG['contract_price_per_kw_month'] * CFG['oc_within_10pct_mult']
        + oc_seg2[mo] * CFG['contract_price_per_kw_month'] * CFG['oc_beyond_10pct_mult']
        for mo in months_in_data
    )
    
    # Degradation (simplified linear for now; PWL extension below)
    deg_cost = gp.quicksum(
        pw_arr[i] * 0.05 * (p_ch[i] + p_dis[i]) for i in range(N))
    
    # T-REC cost
    trec_cost_expr = CFG['trec_cost_per_kwh'] * trec_buy
    
    m.setObjective(inv_cost + grid_cost + bill_penalty + deg_cost + trec_cost_expr,
                   GRB.MINIMIZE)
    
    # ── Solve ──
    print(f'  Variables: {m.NumVars:,} ({m.NumIntVars:,} binary)')
    print(f'  Constraints: {m.NumConstrs:,}')
    t0 = time.time()
    m.optimize()
    solve_time = time.time() - t0
    
    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        print(f'  WARNING: Solver status = {m.Status}')
        return None
    
    # ── Extract results ──
    pv_val = cap_pv.X
    bp_val = cap_bp.X
    be_val = cap_be.X
    cd_val = cap_cd.X
    obj = m.ObjVal
    
    # RE percentage
    re_val = sum(pw_arr[i] * (p_pv_load[i].X + green_dis[i].X) for i in range(N))
    re_pct = re_val / total_load * 100
    trec_val = trec_buy.X * CFG['trec_cost_per_kwh']
    
    print(f'  Objective: {obj:,.0f} TWD ({obj/1e6:.2f}M)')
    print(f'  PV={pv_val:,.0f} kW, BESS={bp_val:,.0f}/{be_val:,.0f} kW/kWh, Contract={cd_val:,.0f} kW')
    print(f'  RE={re_pct:.1f}%, T-REC={trec_buy.X:,.0f} kWh ({trec_val/1e6:.2f}M TWD)')
    print(f'  Solve: {solve_time:.1f}s, Gap: {m.MIPGap*100:.4f}%')
    
    return {
        'cap_pv': pv_val, 'cap_bp': bp_val, 'cap_be': be_val, 'cap_cd': cd_val,
        'obj': obj, 're_pct': re_pct, 'trec_cost': trec_val,
        'solve_time': solve_time, 'gap': m.MIPGap,
    }

## Run All 8 Cases

In [3]:
all_results = []

for case in CASE_TABLE:
    name = case['name']
    print(f"\n{'='*60}")
    print(f"  CASE: {name}")
    print(f"{'='*60}")
    print(f"  Method1={case['method1']}, RiskDays={case['risk_days']}, "
          f"ProbPV={case['prob_pv']}, Uplift={case['uplift']}")
    
    # Load data with case flags
    repday_data, calendar_order, info = load_data(CFG, case)
    
    # Build and solve
    result = build_and_solve(CFG, repday_data, calendar_order, info, case)
    
    if result is not None:
        formatted = format_results(
            name, result['cap_pv'], result['cap_bp'], result['cap_be'], result['cap_cd'],
            result['obj'], result['re_pct'], result['trec_cost'], result['solve_time'], CFG)
        all_results.append(formatted)
    else:
        all_results.append({'case': name, 'status': 'infeasible'})

print(f"\n\n{'='*60}")
print(f"  ALL {len(all_results)} CASES COMPLETE")
print(f"{'='*60}")


  CASE: M0_I0_R0
  Method1=False, RiskDays=False, ProbPV=False, Uplift=None
  Repdays: 16 (16 body + 0 risk)
  Scenarios: 1/repday, 24 hours
  Weight sum: 365
  Calendar days: N/A (no Method 1)
Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2797924


Academic license 2797924 - for non-commercial use only - registered to m1___@mail.ntust.edu.tw


Set parameter OutputFlag to value 1


...

Thread count was 10 (of 10 available processors)


Solution count 10: 8.50049e+07 8.50049e+07 8.50232e+07 ... 8.68538e+07


Optimal solution found (tolerance 1.00e-04)


Best objective 8.500492544100e+07, best bound 8.500408032932e+07, gap 0.0010%


  Objective: 85,004,925 TWD (85.00M)
  PV=7,055 kW, BESS=1,298/8,126 kW/kWh, Contract=3,074 kW
  RE=38.6%, T-REC=0 kWh (0.00M TWD)
  Solve: 43.4s, Gap: 0.0010%


  ALL 8 CASES COMPLETE


## Results Comparison Table

In [4]:
results_df = pd.DataFrame(all_results)

# Display
display_cols = ['case', 'pv_kw', 'bess_p_kw', 'bess_e_kwh', 'ep_ratio',
                'contract_kw', 'total_cost_M', 'invest_M', 'opex_M',
                'trec_M', 're_pct', 'solve_s']
cols = [c for c in display_cols if c in results_df.columns]
print(results_df[cols].to_string(index=False))

# Save
out_path = Path(CFG['output_dir'])
results_df.to_csv(out_path / 'case_summary_main.csv', index=False)

with open(out_path / 'case_results_all.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"\nResults saved to {out_path}")

       case  pv_kw  bess_p_kw  bess_e_kwh  ep_ratio  contract_kw  total_cost_M  invest_M  opex_M  trec_M  re_pct  solve_s
   M0_I0_R0 6955.3     1584.6     11016.2       7.0       2347.6         75.88     39.51   36.37     0.0    40.7      0.2
   M1_I0_R0 6541.4      970.8      4222.7       4.3       2669.9         79.25     32.75   46.49     0.0    34.0      3.3
   M2_I0_R0 7240.5     1501.3      7802.7       5.2       2888.9         82.92     39.14   43.78     0.0    39.1      9.3
   M2_I1_R0 6711.1     1350.0      7899.0       5.9       3026.9         83.52     37.72   45.80     0.0    36.2     25.9
M2_I1_R1_p3 6890.0     1381.1      8049.2       5.8       3126.3         86.03     38.72   47.31     0.0    36.9     55.3
M2_I1_R1_p5 7046.6     1417.5      8293.9       5.9       3178.3         87.70     39.60   48.10     0.0    37.3     44.7
M2_I1_R2_p3 6901.5     1308.4      7932.1       6.1       3064.9         84.41     38.41   46.00     0.0    36.9     37.3
M2_I1_R2_p5 7054.8     1

## Key Comparisons

### Value of Probabilistic Information: M2_I0_R0 vs M2_I1_R0
### Method 1 Gain: M0_I0_R0 vs M1_I0_R0 vs M2_I0_R0
### Robustness: M2_I1_R0 vs R1/R2 variants

In [5]:
if len(results_df) >= 4 and 'total_cost_M' in results_df.columns:
    # Value of probabilistic info
    r_i0 = results_df[results_df['case'] == 'M2_I0_R0']
    r_i1 = results_df[results_df['case'] == 'M2_I1_R0']
    if len(r_i0) > 0 and len(r_i1) > 0:
        cost_diff = r_i0['total_cost_M'].iloc[0] - r_i1['total_cost_M'].iloc[0]
        print(f"Value of probabilistic PV info: {cost_diff:.2f}M TWD")
        print(f"  M2_I0_R0 (deterministic): {r_i0['total_cost_M'].iloc[0]:.2f}M")
        print(f"  M2_I1_R0 (probabilistic): {r_i1['total_cost_M'].iloc[0]:.2f}M")
    
    # Method 1 progression
    print(f"\nMethod 1 progression:")
    for c in ['M0_I0_R0', 'M1_I0_R0', 'M2_I0_R0']:
        row = results_df[results_df['case'] == c]
        if len(row) > 0:
            print(f"  {c}: {row['total_cost_M'].iloc[0]:.2f}M TWD")
    
    # Robustness
    print(f"\nRobustness (load uplift):")
    for c in ['M2_I1_R0', 'M2_I1_R1_p3', 'M2_I1_R1_p5', 'M2_I1_R2_p3', 'M2_I1_R2_p5']:
        row = results_df[results_df['case'] == c]
        if len(row) > 0:
            print(f"  {c}: {row['total_cost_M'].iloc[0]:.2f}M, "
                  f"Contract={row['contract_kw'].iloc[0]:.0f} kW, "
                  f"RE={row['re_pct'].iloc[0]:.1f}%")
else:
    print('Not enough completed cases for comparison')

Value of probabilistic PV info: -0.60M TWD
  M2_I0_R0 (deterministic): 82.92M
  M2_I1_R0 (probabilistic): 83.52M

Method 1 progression:
  M0_I0_R0: 75.88M TWD
  M1_I0_R0: 79.25M TWD
  M2_I0_R0: 82.92M TWD

Robustness (load uplift):
  M2_I1_R0: 83.52M, Contract=3027 kW, RE=36.2%
  M2_I1_R1_p3: 86.03M, Contract=3126 kW, RE=36.9%
  M2_I1_R1_p5: 87.70M, Contract=3178 kW, RE=37.3%
  M2_I1_R2_p3: 84.41M, Contract=3065 kW, RE=36.9%
  M2_I1_R2_p5: 85.00M, Contract=3074 kW, RE=38.6%


## Summary

| Case | Description | Key Question |
|------|-------------|-------------|
| M0_I0_R0 | Plain repdays, det PV, no Method 1 | Baseline distortion |
| M1_I0_R0 | + Method 1 chronology | Method 1 gain |
| M2_I0_R0 | + Risk days | Full bridge, det PV |
| M2_I1_R0 | + Probabilistic PV | **Mainline** — value of stochastic info |
| M2_I1_R1_p3/p5 | + All-day load uplift | Demand growth robustness |
| M2_I1_R2_p3/p5 | + Peak-hour uplift | Contract/over-contract tail risk |